# P9 — sparse seed 1 SAE/TC 16개 전체 평가

학습 반환 감사 통과 후의 **평가 전용** 노트북입니다. 학습을 다시 실행하지 않습니다.
ZIP `v1_4_p9_evaluation_r1.zip`을 내 드라이브 `boolean_interp_v1_4/`에 업로드하고 GPU 런타임에서 위부터 실행하세요.
기존 P5_r2를 읽고 새 결과를 P9_evaluation_r1/sae 및 /tc에 구분해 저장합니다.

LM seed 0 × block 0/3/7/11 × k=4/16, SAE 8개와 TC 8개입니다. 모든 train/val 선택을 동결한 뒤 test를 평가합니다.
SAE의 h 복원과 TC의 u→m 예측을 구분하며 TC 출력 패칭에는 s_m을 곱하고 원본 residual skip을 유지합니다.

중단 후 같은 셀을 재실행하면 완료 fit/선택/pair 단위부터 재개합니다. 다른 환경은 새 환경 ID와 GPU smoke를 기록합니다.
실행 로그와 단계별 시간을 Drive에 저장합니다. 실패·NA·matching coverage를 보존하며 test로 설정을 바꾸지 않습니다.
마지막 ZIP·checksum을 반환한 뒤 원본 cache 독립 재현 감사를 거쳐야 P9 완료를 판단합니다.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib,zipfile,subprocess,sys,time,json,os,shutil
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/boolean_interp_v1_4')
bundle=Path('/content/v1_4_p9_evaluation_r1.zip')
def digest(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
EXPECTED='3ce8f0f01b54930ffcad9a4c56c5f2f601e31ebad86de21fafe7b08bbfcacbad'
if not bundle.exists() or digest(bundle)!=EXPECTED:
    shutil.copyfile(BASE/bundle.name,bundle)
assert digest(bundle)==EXPECTED, '입력 ZIP checksum 불일치'
ROOT=Path('/content/MI_P9_evaluation_r1');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    for i in z.infolist():assert (ROOT/i.filename).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
SOURCE=BASE/'P5_r2';OUTPUT=BASE/'P9_evaluation_r1';OUTPUT.mkdir(parents=True,exist_ok=True)
assert (SOURCE/'contract.json').exists(), 'P5 원본 경로 확인 필요'


## 1. 의존성·입력 검증
설치 로그·환경 lock·GPU 정보를 보존합니다. 런타임 재시작이 필요하면 재시작 후 첫 셀부터 실행하세요.

In [ ]:
log=subprocess.run([sys.executable,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pytest==8.4.2','matplotlib==3.10.8','torch==2.11.0','--extra-index-url','https://download.pytorch.org/whl/cu128'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(OUTPUT/f'install_{time.time_ns()}.txt').write_text(log.stdout);print(log.stdout);log.check_returncode()
os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='2';sys.path.insert(0,str(ROOT))
from interp_v1_4 import p9_sae_evaluation,p9_tc_evaluation
for ev in (p9_sae_evaluation,p9_tc_evaluation):ev.verify(ROOT)
def logged(command):
    logs=OUTPUT/'logs';logs.mkdir(exist_ok=True)
    path=logs/f'command_{time.time_ns()}.log';print('실행:', ' '.join(command),flush=True)
    with path.open('w',buffering=1) as f:
        f.write(json.dumps(dict(command=command,started_ns=time.time_ns()))+'\n')
        proc=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in proc.stdout:f.write(line);print(line,end='',flush=True)
            rc=proc.wait();f.write(json.dumps(dict(returncode=rc,ended_ns=time.time_ns()))+'\n')
            if rc:raise subprocess.CalledProcessError(rc,command)
        finally:
            if proc.poll() is None:
                proc.terminate()
                try:proc.wait(timeout=10)
                except subprocess.TimeoutExpired:proc.kill();proc.wait()
            proc.stdout.close()
def stage(tool,action,*extra):
    logged([sys.executable,'-u','-m',f'interp_v1_4.p9_{tool}_runner',action,'--root',str(ROOT),'--source',str(SOURCE),'--output',str(OUTPUT/tool),*extra])
logged([sys.executable,'-m','pytest','tests_v1_4/test_p9_sae_evaluation.py','tests_v1_4/test_p9_tc_evaluation.py','tests_v1_4/test_p9_evaluation_contract.py','-q'])


## 2. 현재 환경 CUDA smoke
두 도구의 네 층·두 k에서 identity·전체 donor·평균·근사·선택·random latent·좌표·random 방향과 양방향 changed/unchanged 경로를 확인합니다.

In [ ]:
SMOKES={}
for tool in ('sae','tc'):
    folder=OUTPUT/tool/'smoke'/str(time.time_ns());SMOKES[tool]=folder/'smoke.json'
    logged([sys.executable,'-u','-m',f'interp_v1_4.p9_{tool}_runner','smoke','--root',str(ROOT),'--output',str(folder),'--device','cuda'])


## 3. train/validation 선택 동결
Seed 0 원본 cache checksum·READ key를 대조합니다. P5의 검증된 full probe와 block 0 대조군을 재사용하고 나머지 train/val probe를 적합합니다. 각 dictionary별 단일/≤4 feature, 전체·128후보, 현재/과거/상태·보조 label과 감독 변수 전이를 독립 선택합니다. 모든 선택을 마친 뒤 다음 셀로 진행합니다.

In [ ]:
for tool in ('sae','tc'):stage(tool,'select','--device','cuda')


## 4. 의미·fidelity 평가
Balanced accuracy/F1/AUROC·sequence bootstrap CI, 현재≠과거 subset·변수 전이, NMSE/R2/EV·L0와 train 전체 재인코딩 dead 비율을 저장합니다. Test는 선택에 사용하지 않습니다.

In [ ]:
for tool in ('sae','tc'):stage(tool,'semantic','--device','cuda')


## 5. 한 READ 위치 대체·인과 평가
Run마다 causal val norm bins를 고정하고 2,048개의 shared general READ target 및 changed/unchanged 각 1,024 pairs를 양방향으로 평가합니다. 두 feature 규모와 모든 대조군을 실행하고 pair별로 저장합니다. 동일 latent ID를 dictionary 간 동일 feature로 간주하지 않습니다.

In [ ]:
for tool in ('sae','tc'):stage(tool,'patch','--device','cuda','--smoke',str(SMOKES[tool]))


## 6. 전체 표·그림·원시 결과 검산·초기화 비교
각 도구 8개 모두 완료해야 집계합니다. 층간 paired CI와 기존 seed 0 결과에 대한 초기화 민감도 표·그림을 만듭니다. Seed 차이는 LM seed 0 안의 점 추정치 비교로 표시하며 독립 LM 반복으로 취급하지 않습니다. Full fitting/fidelity/CI 원본 재현 감사는 반환 후 별도로 수행합니다.

In [ ]:
for tool in ('sae','tc'):
    stage(tool,'aggregate')
    logged([sys.executable,'-u',str(ROOT/f'scripts/verify_v1_4_p9_{tool}_evaluation.py'),'--root',str(ROOT),'--output',str(OUTPUT/tool)])
logged([sys.executable,'-u','-m','interp_v1_4.p9_comparison','--root',str(ROOT),'--output',str(OUTPUT)])


## 7. 결과 ZIP·checksum 다운로드
완료 또는 중단 후 실행합니다. 두 도구의 결과·환경·실패·재개 기록과 초기화 비교를 하나의 ZIP으로 반환합니다. 원본 P5 cache와 입력 모델은 중복 포함하지 않습니다. 다운로드한 두 파일을 이 작업에 전달하세요.

In [ ]:
archive=Path(f'/content/v1_4_p9_evaluation_return_{time.time_ns()}.zip')
from interp_v1_4.p9_sae_runner import export
export(OUTPUT,archive)
print('SHA256:',digest(archive))
files.download(str(archive));files.download(str(archive.with_suffix('.sha256.json')))
